# 07 Post Content Clustering
Cluster posts via K-Means, GMM, and optional HDBSCAN.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Feature Matrix

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


In [3]:
import importlib.util
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
has_hdbscan = importlib.util.find_spec("hdbscan") is not None
if has_hdbscan:
    import hdbscan

feat = ["engagement_rate","view_rate","comment_rate","like_rate","view_engagement_rate","caption_length","hashtags_count","emoji_count","discount_percent","posting_hour"]
X = StandardScaler().fit_transform(df[feat].fillna(0))

## Experiments and Metric Comparison

In [4]:
rows, labels_store = [], {}
for k in range(3,11):
    y = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    rows.append({"model":"kmeans","setting":f"k={k}","silhouette":silhouette_score(X,y),"davies_bouldin":davies_bouldin_score(X,y),"calinski_harabasz":calinski_harabasz_score(X,y),"clusters":len(np.unique(y)),"noise_ratio":0.0})
    labels_store[("kmeans",f"k={k}")] = y

for k in range(3,11):
    for cov in ["full","tied","diag"]:
        y = GaussianMixture(n_components=k, covariance_type=cov, random_state=42).fit_predict(X)
        if len(np.unique(y)) < 2:
            continue
        rows.append({"model":"gmm","setting":f"components={k},cov={cov}","silhouette":silhouette_score(X,y),"davies_bouldin":davies_bouldin_score(X,y),"calinski_harabasz":calinski_harabasz_score(X,y),"clusters":len(np.unique(y)),"noise_ratio":0.0})
        labels_store[("gmm",f"components={k},cov={cov}")] = y

if has_hdbscan:
    for mcs in [5,10,20]:
        y = hdbscan.HDBSCAN(min_cluster_size=mcs).fit_predict(X)
        valid = y >= 0
        if len(np.unique(y[valid])) < 2:
            continue
        rows.append({"model":"hdbscan","setting":f"min_cluster_size={mcs}","silhouette":silhouette_score(X[valid],y[valid]),"davies_bouldin":davies_bouldin_score(X[valid],y[valid]),"calinski_harabasz":calinski_harabasz_score(X[valid],y[valid]),"clusters":len(np.unique(y[valid])),"noise_ratio":float((y==-1).mean())})
        labels_store[("hdbscan",f"min_cluster_size={mcs}")] = y

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["silhouette","calinski_harabasz","clusters"], lower_is_better_cols=["davies_bouldin","noise_ratio"])
best = exp.iloc[0]
y = labels_store[(best["model"], best["setting"])]
out = df.copy()
out["cluster_id"] = y
profiles = out[out["cluster_id"] >= 0].groupby("cluster_id", as_index=False).agg(
    posts=("cluster_id","size"),
    engagement_rate=("engagement_rate","mean"),
    view_rate=("view_rate","mean"),
    comment_rate=("comment_rate","mean"),
    top_post_type=("post_type", lambda x: x.mode().iat[0]),
    top_time=("posting_time_bin", lambda x: x.mode().iat[0]),
)
labels = ["high-engagement storytellers","reach-focused visuals","discussion starters","promotion-heavy posts","steady baseline content","niche audience posts","community updates","experimental formats"]
profiles = profiles.sort_values("engagement_rate", ascending=False).reset_index(drop=True)
profiles["cluster_label"] = [labels[i] if i < len(labels) else f"cluster_{i}" for i in range(len(profiles))]
out = out.merge(profiles[["cluster_id","cluster_label"]], on="cluster_id", how="left")

  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executabl

## Save Outputs

In [5]:
out.to_csv(PROCESSED_DIR / "post_clusters.csv", index=False)
profiles.to_csv(PROCESSED_DIR / "cluster_profiles.csv", index=False)
exp.to_csv(REPORTS_DIR / "post_clustering_experiments.csv", index=False)
display(exp.head(15))
display(profiles)
print("Insight: cluster archetypes support reusable content playbooks.")

,model,setting,silhouette,davies_bouldin,calinski_harabasz,clusters,noise_ratio,composite_score
0,kmeans,k=10,0.146411,1.570437,154.023584,10,0.0,4.032331
1,kmeans,k=9,0.154738,1.530454,163.425380,9,0.0,4.013419
2,kmeans,k=8,0.148224,1.536631,172.988594,8,0.0,3.880113
3,kmeans,k=7,0.156731,1.543217,183.402831,7,0.0,3.837351
4,kmeans,k=3,0.200616,1.827511,264.941108,3,0.0,3.767804
5,kmeans,k=6,0.153247,1.633142,193.869292,6,0.0,3.671813
6,kmeans,k=4,0.179740,1.742012,225.678859,4,0.0,3.637046
7,kmeans,k=5,0.151244,1.761690,206.695194,5,0.0,3.502241
8,gmm,"components=3,cov=diag",0.177352,1.880546,242.009637,3,0.0,3.477456
9,gmm,"components=10,cov=tied",0.118611,1.866375,114.947532,10,0.0,3.471929


,cluster_id,posts,engagement_rate,view_rate,comment_rate,top_post_type,top_time,cluster_label
0,6,29,0.613364,4.062143,0.052822,reel,night,high-engagement storytellers
1,4,93,0.358517,2.699720,0.029792,reel,evening,reach-focused visuals
2,1,79,0.197042,2.853575,0.016181,reel,afternoon,discussion starters
3,2,120,0.144450,0.980366,0.013093,image,afternoon,promotion-heavy posts
4,5,113,0.113922,1.073512,0.010811,image,afternoon,steady baseline content
5,8,61,0.084943,0.835301,0.007439,image,afternoon,niche audience posts
6,3,98,0.083834,0.898307,0.006499,image,afternoon,community updates
7,0,158,0.067906,0.857944,0.005560,image,night,experimental formats
8,9,100,0.055056,0.588607,0.004175,image,night,cluster_8
9,7,149,0.046971,0.719145,0.003541,image,afternoon,cluster_9


Insight: cluster archetypes support reusable content playbooks.
